# T05 CP3 | Online Retail II: Cleaning, EDA, Bronze, Silver

**Team 5:** Boomikaa P, Chahatleen Kaur, Saba Gul, Vanshika Malviya, K. Shreya, Tushar Rathor.

This notebook follows the requested order: prepare a cleaned working copy, explore it with EDA, then save the raw Bronze layer and cleaned Silver tables for checkpoint 3. Bronze preserves the original source rows; cleaning is applied to a separate copy. Gold tables and a dashboard belong to checkpoint 4.

Dataset: Chen (2012), UCI Online Retail II, CC BY 4.0, https://doi.org/10.24432/C5CG6D. Export both Excel sheets as CSV and upload them to the configured Unity Catalog Volume.

## 1. Set paths and table names
The catalog, schema, and volume below use the requested names. A schema is a named group of tables; a volume is a managed place for input files.

In [ ]:
from pyspark.sql import functions as F

CATALOG = "retail_catalog"
SCHEMA = "team5_retail"
VOLUME = "retail_files"
BASE = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
FILES = [
    f"{BASE}/online_retail_2009_2010.csv",
    f"{BASE}/online_retail_2010_2011.csv",
]
DB = f"{CATALOG}.{SCHEMA}"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")


## 2. Load the source CSVs
We read both sheets as text and keep a raw copy. A Spark DataFrame is a table Spark can process across the Databricks compute. The source file column records data lineage: where each row came from.

In [ ]:
parts = []
for file_path in FILES:
    part = (spark.read.option("header", True).option("mode", "PERMISSIVE")
            .csv(file_path)
            .withColumn("source_file", F.lit(file_path)))
    parts.append(part)
raw = parts[0].unionByName(parts[1], allowMissingColumns=True)
raw.printSchema()
raw.show(10, truncate=False)


## 3. Clean a working copy before EDA
We keep `raw` unchanged so Bronze can preserve the original data. In the separate `cleaned` copy, we standardize text and types, remove exact duplicate rows, flag missing or unusual values, and mark cancellation invoices. We do not remove missing customer IDs, negative quantities, or nonpositive prices.

In [ ]:
renames = [("Invoice", "invoice"), ("StockCode", "stock_code"),
           ("Description", "description"), ("Quantity", "quantity"),
           ("InvoiceDate", "invoice_date"), ("Price", "price"),
           ("Customer ID", "customer_id"), ("Country", "country")]
cleaned = raw
for old_name, new_name in renames:
    cleaned = cleaned.withColumnRenamed(old_name, new_name)

cleaned = (cleaned
    .withColumn("invoice", F.trim("invoice"))
    .withColumn("stock_code", F.upper(F.trim("stock_code")))
    .withColumn("description", F.trim(F.regexp_replace("description", r"\s+", " ")))
    .withColumn("country", F.initcap(F.trim("country")))
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("customer_id", F.col("customer_id").cast("long"))
    .withColumn("invoice_date", F.coalesce(
        F.to_timestamp("invoice_date", "M/d/yyyy H:mm"),
        F.to_timestamp("invoice_date")))
    .dropDuplicates()
    .withColumn("is_cancelled", F.upper(F.col("invoice")).startswith("C"))
    .withColumn("missing_customer_id", F.col("customer_id").isNull())
    .withColumn("zero_or_negative_price", F.col("price").isNull() | (F.col("price") <= 0))
    .withColumn("negative_quantity", F.col("quantity") < 0)
    .withColumn("missing_description", F.col("description").isNull() | (F.col("description") == "")))

print("Rows read:", raw.count())
print("Rows after exact duplicate removal:", cleaned.count())


## 4. EDA on the cleaned working copy
**Exploratory data analysis (EDA)** means checking the data before making business conclusions. We look at row counts, unique values, missing values, date and number ranges, cancellations, and countries. The EDA uses the cleaned copy, so the duplicate count above shows the effect of cleaning.

In [ ]:
profile = cleaned.agg(
    F.count("*").alias("rows_after_cleaning"),
    F.countDistinct("invoice").alias("distinct_invoices"),
    F.countDistinct("stock_code").alias("distinct_stock_codes"),
    F.countDistinct("customer_id").alias("distinct_customers"),
    F.countDistinct("country").alias("distinct_countries"),
    F.sum(F.col("missing_customer_id").cast("long")).alias("missing_customer_id_rows"),
    F.sum(F.col("missing_description").cast("long")).alias("blank_description_rows"),
    F.sum(F.col("is_cancelled").cast("long")).alias("cancellation_rows"),
    F.sum(F.col("negative_quantity").cast("long")).alias("negative_quantity_rows"),
    F.sum(F.col("zero_or_negative_price").cast("long")).alias("zero_or_negative_price_rows"),
    F.min("invoice_date").alias("first_invoice_date"),
    F.max("invoice_date").alias("last_invoice_date"),
    F.min("quantity").alias("minimum_quantity"), F.max("quantity").alias("maximum_quantity"),
    F.min("price").alias("minimum_price"), F.max("price").alias("maximum_price")
).first()
for name, value in profile.asDict().items():
    print(f"{name}: {value}")
if profile["rows_after_cleaning"]:
    missing_rate = 100 * profile["missing_customer_id_rows"] / profile["rows_after_cleaning"]
    print("missing customer ID rate (%):", round(missing_rate, 2))

print("Top countries by row count:")
cleaned.groupBy("country").count().orderBy(F.desc("count")).show(10, truncate=False)
print("Number summary:")
cleaned.select("quantity", "price").describe().show()


## 5. Bronze: save the unchanged source rows
**Bronze** is the raw stage of the lakehouse. A **lakehouse** stores data in managed tables for later processing and analysis. A **Delta table** is Databricks’ managed table format. We add only a load timestamp to the raw rows.

In [ ]:
bronze = raw.withColumn("loaded_at", F.current_timestamp())
bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DB}.bronze_retail_raw")
print("Bronze raw rows:", raw.count())


## 6. Silver: save cleaned sales and cancellations separately
**Silver** is the cleaned stage. Cancellation invoices are stored separately. Regular sales retain flags for missing customers, blank descriptions, negative quantities, and nonpositive prices so the issues stay visible.

In [ ]:
cleaned_cancellations = cleaned.filter(F.col("is_cancelled"))
cleaned_sales = cleaned.filter(~F.col("is_cancelled"))
cleaning_exceptions = cleaned_sales.filter(
    F.col("stock_code").isNull() | (F.col("stock_code") == "") |
    F.col("invoice_date").isNull() | F.col("quantity").isNull() | F.col("price").isNull())

cleaned_cancellations.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DB}.silver_cancellations")
cleaned_sales.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DB}.silver_retail_sales")
cleaning_exceptions.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DB}.silver_quality_exceptions")


## 7. Save a data quality log
A **data quality check** measures whether records are complete and sensible. This small log records the raw and cleaned counts and important issue counts. Review the log before using the tables.

In [ ]:
quality_rows = [
    ("raw_input_rows", str(raw.count())),
    ("rows_after_exact_duplicate_removal", str(cleaned.count())),
    ("cancellation_rows", str(cleaned_cancellations.count())),
    ("silver_sales_rows", str(cleaned_sales.count())),
    ("sales_missing_customer_id_rows", str(cleaned_sales.filter("missing_customer_id").count())),
    ("sales_negative_quantity_rows", str(cleaned_sales.filter("negative_quantity").count())),
    ("sales_zero_or_negative_price_rows", str(cleaned_sales.filter("zero_or_negative_price").count())),
    ("sales_quality_exception_rows", str(cleaning_exceptions.count())),
]
quality_log = spark.createDataFrame(quality_rows, ["check_name", "check_value"])
quality_log = quality_log.withColumn("logged_at", F.current_timestamp())
quality_log.write.format("delta").mode("overwrite").saveAsTable(f"{DB}.silver_quality_log")
quality_log.show(truncate=False)


## Notes

The provided workbook contains 1,067,371 rows across 1 Dec 2009 to 9 Dec 2011. A full local profile found missing customer IDs, repeated rows, cancellations, negative quantities, and zero/negative prices. The notebook calculates its own EDA from your uploaded CSVs.

Test on a small sample first, then run the full dataset once to respect the Databricks Free Edition quota. This notebook requires Databricks, PySpark, and a Unity Catalog Volume. It is not intended for plain local Python. The workflow for this checkpoint is clean working copy → EDA → raw Bronze → cleaned Silver; Gold tables and a dashboard are checkpoint 4.